In [ ]:
"""HypothesisLab_Personas (1).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/186A77B2TN8yzQ2RyWkH_CXifPa-8srkc

# Hypothesis Lab — 8 синтетических персон → топ‑3 → 9 гипотез

Ноутбук реализует новый контур Hypothesis Lab:

1. пользователь загружает JSON с продуктом или запускает демо «Комбо Цифра карта»;
2. `Generation Readiness Validator` проверяет полноту входа и выводит уточняющие вопросы;
3. восемь заранее заданных синтетических персон отвечают на единый набор исследовательских вопросов;
4. LLM отдельно оценивает каждую персону по пяти метрикам;
5. Python применяет веса, проверяет обязательные критерии продукта и выбирает топ‑3;
6. для каждой персоны из топ‑3 создаются ровно три разные маркетинговые гипотезы;
7. Python проверяет ссылки на продуктовые факты, разнообразие и запрещённые обещания.
"""

In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes "pydantic>=2.7" pandas json-repair

In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

In [ ]:
import pandas as pd
import torch
from IPython.display import display
from json_repair import repair_json
from pydantic import BaseModel, Field

In [ ]:
print("CUDA доступна:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не доступна. В Colab выберите: "
        "Среда выполнения → Сменить среду выполнения → GPU."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
"""## 1. Загрузка Qwen3‑4B

Модель работает локально в среде Colab. `enable_thinking=False` упрощает получение валидного JSON.

"""

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B"
RANDOM_SEED = 42

In [ ]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
)
model.eval()

In [ ]:
print("Модель загружена:", MODEL_NAME)

In [ ]:
"""## 2. Схемы данных

`ProductFact` хранит только подтверждённые свойства продукта. Свободный текст гипотезы не считается доказательством: каждая гипотеза обязана ссылаться на `fact_id`, а точные формулировки фактов добавляет Python.

"""

In [ ]:
class ProductFact(BaseModel):
    fact_id: str
    text: str

In [ ]:
class EligibilityRule(BaseModel):
    rule_id: str
    field: str
    operator: Literal["eq", "in", "gte", "lte", "between", "truthy"]
    value: str | int | bool | list[str] | list[int]
    description: str

In [ ]:
class Product(BaseModel):
    brand: str
    name: str
    description: str
    category: str
    geography: str
    experiment_goal: str
    target_action: str
    source_url: str | None = None
    facts: list[ProductFact] = Field(min_length=3)
    eligibility_rules: list[EligibilityRule] = []
    allowed_channels: list[str] = Field(min_length=1)
    forbidden_claims: list[str] = []
    known_constraints: list[str] = []

In [ ]:
class SyntheticPersona(BaseModel):
    persona_id: str
    label: str
    gender: Literal["женщина", "мужчина"]
    age: int = Field(ge=18, le=80)
    monthly_income_rub: int = Field(gt=0)
    occupation: str
    employment_type: str
    citizenship: str
    permanent_registration_country: str
    device: str
    operating_system: str
    digital_behavior: str
    interests: list[str] = Field(min_length=2)
    geography: str
    cifra_relationship: str
    has_cifra_debit_or_salary_card: bool
    financial_behavior: str
    credit_attitude: str

In [ ]:
class InterviewQuestion(BaseModel):
    question_id: str
    dimension: Literal[
        "problem", "pain", "need", "desire", "objection",
        "trigger", "decision_factor", "use_case", "channel", "action"
    ]
    text: str

In [ ]:
class InterviewAnswer(BaseModel):
    question_id: str
    answer: str

In [ ]:
class PersonaResearch(BaseModel):
    persona_id: str
    interview_answers: list[InterviewAnswer] = Field(min_length=10, max_length=10)
    problems: list[str] = Field(min_length=2, max_length=5)
    pains: list[str] = Field(min_length=2, max_length=5)
    needs: list[str] = Field(min_length=2, max_length=5)
    desires: list[str] = Field(min_length=2, max_length=5)
    objections: list[str] = Field(min_length=1, max_length=4)
    decision_triggers: list[str] = Field(min_length=1, max_length=4)
    decision_factors: list[str] = Field(min_length=2, max_length=5)
    relevant_use_cases: list[str] = Field(min_length=1, max_length=4)
    best_message_angle: str
    fit_summary: str

In [ ]:
class PersonaMetricScores(BaseModel):
    persona_id: str
    need_strength: int = Field(ge=0, le=100)
    product_value_fit: int = Field(ge=0, le=100)
    adoption_readiness: int = Field(ge=0, le=100)
    digital_access_fit: int = Field(ge=0, le=100)
    hypothesis_testability: int = Field(ge=0, le=100)
    strongest_signal: str
    main_barrier: str
    evidence: list[str] = Field(min_length=2, max_length=5)
    explanation: str

In [ ]:
class HypothesisDraft(BaseModel):
    hypothesis_id: str
    persona_id: str
    insight_type: Literal["pain", "need", "desire", "objection", "trigger", "use_case"]
    insight_basis: list[str] = Field(min_length=1, max_length=3)
    value_angle: Literal[
        "safety_net", "control", "one_card", "flexibility",
        "convenience", "clarity", "planned_expense"
    ]
    hypothesis_statement: str
    headline: str
    message_frame: str
    cta: str
    channel: str
    selected_fact_ids: list[str] = Field(min_length=1, max_length=3)
    experiment_design: str
    primary_metric: str
    why_for_persona: str
    risks: list[str] = Field(min_length=1, max_length=3)

In [ ]:
class HypothesisBatch(BaseModel):
    hypotheses: list[HypothesisDraft] = Field(min_length=3, max_length=3)

In [ ]:
"""## 3. Общая функция вызова LLM

Полная Pydantic Schema не передаётся модели: Qwen3‑4B иногда копирует `$defs`
вместо ответа. Модель получает только список верхнеуровневых полей, а конкретный
формат задаётся компактным примером в промпте каждого модуля.

Парсер:

- выделяет несколько JSON-объектов, если модель напечатала их подряд;
- проверяет каждый объект по Pydantic и пропускает случайно повторённую схему;
- исправляет небольшие синтаксические ошибки с помощью `json-repair`;
- при неудаче просит вернуть весь объект заново без Markdown и `$defs`.

"""

In [ ]:
def split_json_object_fragments(text: str) -> list[str]:
    """Выделяет сбалансированные верхнеуровневые {...} с учетом строк."""
    fragments = []
    start = None
    depth = 0
    in_string = False
    escaped = False

    for index, char in enumerate(text):
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            if depth == 0:
                start = index
            depth += 1
        elif char == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start is not None:
                fragments.append(text[start:index + 1])
                start = None

    return fragments

In [ ]:
def extract_json_candidates(text: str) -> list[dict]:
    """Возвращает все похожие на ответ JSON-объекты, включая repaired."""
    cleaned = text.strip()
    cleaned = re.sub(r"```(?:json)?", "", cleaned, flags=re.IGNORECASE)
    fragments = split_json_object_fragments(cleaned)

    # Если объект оборван, json-repair иногда может восстановить весь хвост.
    if not fragments and "{" in cleaned:
        fragments = [cleaned[cleaned.find("{"):]]

    candidates = []
    seen = set()

    for fragment in fragments:
        parsed_variants = []
        try:
            parsed_variants.append(json.loads(fragment))
        except Exception:
            pass

        try:
            repaired = repair_json(fragment, return_objects=True)
            parsed_variants.append(repaired)
        except Exception:
            pass

        for item in parsed_variants:
            if not isinstance(item, dict):
                continue
            marker = json.dumps(item, ensure_ascii=False, sort_keys=True)
            if marker not in seen:
                seen.add(marker)
                candidates.append(item)

    return candidates

In [ ]:
def run_llm_json(
    system_prompt: str,
    user_prompt: str,
    response_model,
    max_attempts: int = 3,
    temperature: float = 0.20,
    max_new_tokens: int = 1600,
    retry_feedback: str | None = None,
):
    last_error = retry_feedback
    expected_fields = list(response_model.model_fields.keys())
    response_schema = response_model.model_json_schema()
    format_note = (
        "\n\nТЕХНИЧЕСКИЙ ФОРМАТ: верни ровно ОДИН JSON-объект без Markdown, "
        "пояснений и JSON Schema. Не возвращай ключи $defs, properties, title, "
        "required или type на верхнем уровне. Ожидаемые верхнеуровневые поля: "
        + ", ".join(expected_fields)
        + ". Строго соблюдай типы, enum и границы массивов из схемы результата:\n"
        + json.dumps(response_schema, ensure_ascii=False)
    )

    def build_retry_note(error: object) -> str:
        short_error = re.sub(r"\s+", " ", str(error))[:500]
        return (
            "\n\nПРЕДЫДУЩИЙ ОТВЕТ НЕ ПРОШЕЛ ПРОВЕРКУ. "
            "Сформируй ВЕСЬ ответ заново как один JSON-объект. "
            "Не копируй схему и не продолжай предыдущий текст. "
            f"Краткая ошибка: {short_error}"
        )

    retry_note = build_retry_note(retry_feedback) if retry_feedback else ""

    for attempt in range(1, max_attempts + 1):
        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    user_prompt
                    + format_note
                    + retry_note
                ),
            },
        ]
        chat_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.8,
            top_k=20,
        )
        output_ids = generated_ids[0][model_inputs.input_ids.shape[-1]:]
        raw_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

        try:
            candidates = extract_json_candidates(raw_text)
            if not candidates:
                raise ValueError("В ответе модели не найден JSON-объект.")

            candidate_errors = []
            for candidate in candidates:
                try:
                    return response_model.model_validate(candidate)
                except Exception as candidate_exc:
                    candidate_errors.append(candidate_exc)

            raise candidate_errors[-1]

        except Exception as exc:
            last_error = exc
            retry_note = build_retry_note(exc)
            print(f"Попытка {attempt}: JSON не прошёл валидацию — {exc}")
        finally:
            del model_inputs, generated_ids, output_ids
            torch.cuda.empty_cache()

    raise RuntimeError(
        f"После {max_attempts} попыток не удалось получить валидный JSON. "
        f"Последняя ошибка: {last_error}"
    )

In [ ]:
"""## 4. Загрузка продукта и Readiness Validator

По умолчанию запускается демонстрационный продукт — «Комбо Цифра карта». Его факты взяты с официальной страницы банка: [cifra-bank.ru/combo](https://cifra-bank.ru/combo/).

Чтобы использовать свой продукт:

1. переключите `USE_DEMO_PRODUCT = False`;
2. запустите ячейку;
3. загрузите JSON, соответствующий схеме `Product`.

Входной валидатор не дополняет отсутствующие факты сам. Он выдаёт вопросы, на которые должен ответить пользователь до запуска генерации.

"""

In [ ]:
DEMO_PRODUCT = Product(
    brand="Цифра банк",
    name="Комбо Цифра карта",
    description=(
        "Дебетовая и кредитная карта в одной: кредитный лимит подключается "
        "к дебетовой или зарплатной карте Цифра банка через мобильное приложение."
    ),
    category="banking_card",
    geography="Россия",
    experiment_goal=(
        "Выбрать три наиболее перспективные персоны и сформировать "
        "для каждой три маркетинговые гипотезы подключения кредитного лимита."
    ),
    target_action="Подать заявление на подключение кредитного лимита в приложении",
    source_url="https://cifra-bank.ru/combo/",
    facts=[
        ProductFact(
            fact_id="F1",
            text="Комбо Цифра карта совмещает дебетовую и кредитную карту в одной карте.",
        ),
        ProductFact(
            fact_id="F2",
            text=(
                "Кредитный лимит можно подключить к дебетовой или зарплатной "
                "карте Цифра банка через мобильное приложение."
            ),
        ),
        ProductFact(
            fact_id="F3",
            text=(
                "Беспроцентный период длится 55 дней при своевременной уплате "
                "обязательных платежей и полном погашении долга до его окончания."
            ),
        ),
        ProductFact(
            fact_id="F4",
            text=(
                "В заявлении клиент выбирает желаемый кредитный лимит "
                "от 30 000 до 100 000 рублей; решение принимает банк."
            ),
        ),
        ProductFact(
            fact_id="F5",
            text=(
                "Минимальный платеж составляет 5% от суммы основного долга "
                "на конец расчетного периода, но не менее 300 рублей."
            ),
        ),
        ProductFact(
            fact_id="F6",
            text="Заявление и настройка лимита выполняются в мобильном приложении Цифра банка.",
        ),
        ProductFact(
            fact_id="F7",
            text=(
                "Кредитный лимит используется при отсутствии или недостаточности "
                "собственных средств на счетах, привязанных к карте."
            ),
        ),
    ],
    eligibility_rules=[
        EligibilityRule(
            rule_id="E1", field="citizenship", operator="eq", value="Россия",
            description="Гражданство РФ",
        ),
        EligibilityRule(
            rule_id="E2", field="permanent_registration_country", operator="eq", value="Россия",
            description="Постоянная регистрация на территории РФ",
        ),
        EligibilityRule(
            rule_id="E3", field="age", operator="between", value=[21, 70],
            description="Возраст от 21 до 70 лет на дату возврата кредита",
        ),
        EligibilityRule(
            rule_id="E4", field="employment_type", operator="eq", value="постоянная работа",
            description="Постоянное основное место работы",
        ),
        EligibilityRule(
            rule_id="E5", field="operating_system", operator="in", value=["iOS", "Android"],
            description="Смартфон с поддерживаемым приложением на iOS или Android",
        ),
    ],
    allowed_channels=["in_app_banner", "push", "email", "landing", "sms"],
    forbidden_claims=[
        "гарантированное одобрение",
        "одобрение для всех",
        "без проверки",
        "без обязательных платежей",
        "без переплаты при любых условиях",
        "бесплатные деньги",
    ],
    known_constraints=[
        "Для подключения лимита нужна дебетовая или зарплатная карта Цифра банка.",
        "Льготный период требует своевременных обязательных платежей и полного погашения.",
        "Окончательное решение по кредитному лимиту принимает банк.",
    ],
)

In [ ]:
def load_product() -> Product:
    if USE_DEMO_PRODUCT:
        print("Используется демо-продукт:", DEMO_PRODUCT.name)
        return DEMO_PRODUCT

    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Загрузите ровно один JSON-файл с продуктом.")
    filename, content = next(iter(uploaded.items()))
    print("Загружен продукт:", filename)
    return Product.model_validate(json.loads(content.decode("utf-8")))

In [ ]:
def product_readiness_questions(product: Product) -> list[str]:
    questions = []
    if len(product.facts) < 3:
        questions.append("Какие минимум три свойства продукта подтверждены источником?")
    if not product.experiment_goal.strip():
        questions.append("Какую бизнес-гипотезу или цель эксперимента нужно проверить?")
    if not product.target_action.strip():
        questions.append("Какое целевое действие должен совершить пользователь?")
    if not product.allowed_channels:
        questions.append("В каких каналах разрешено тестировать коммуникацию?")
    if not product.known_constraints:
        questions.append("Какие ограничения, комиссии и обязательные условия нужно учесть?")
    if not product.forbidden_claims:
        questions.append("Какие формулировки запрещены юридически или бренд-политикой?")
    if not product.geography.strip():
        questions.append("В какой географии доступен продукт?")
    return questions

In [ ]:
USE_DEMO_PRODUCT = True
product = load_product()
readiness_questions = product_readiness_questions(product)

In [ ]:
if readiness_questions:
    print("Входных данных недостаточно. Ответьте на вопросы:")
    for index, question in enumerate(readiness_questions, start=1):
        print(f"{index}. {question}")
    raise ValueError("Generation Readiness Validator: NOT READY")

In [ ]:
print("Generation Readiness Validator: READY")
print(product.model_dump_json(indent=2))

In [ ]:
"""## 5. Восемь синтетических персон

Профили специально различаются по полу, возрасту, доходу, профессии, устройству, цифровому поведению, отношениям с банком и интересам. В интересах представлены, в частности, автомобиль, телевидение/сериалы и фитнес.

Это фиксированные исследовательские допущения. LLM не меняет демографию и не добавляет имена, работодателей или события из прошлого.

"""

In [ ]:
PERSONAS = [
    SyntheticPersona(
        persona_id="persona_001",
        label="Мобильная специалистка с регулярными расходами",
        gender="женщина", age=27, monthly_income_rub=95_000,
        occupation="маркетолог", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="iPhone 14", operating_system="iOS",
        digital_behavior="почти все финансовые операции делает в приложениях",
        interests=["фитнес", "сериалы", "путешествия"], geography="Москва",
        cifra_relationship="есть дебетовая карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="ведет бюджет, но иногда переносит крупные покупки до зарплаты",
        credit_attitude="готова к короткому льготному кредитованию, боится скрытых условий",
    ),
    SyntheticPersona(
        persona_id="persona_002",
        label="Автомобилист — IT-аналитик",
        gender="мужчина", age=35, monthly_income_rub=180_000,
        occupation="системный аналитик", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="Samsung Galaxy S24", operating_system="Android",
        digital_behavior="активно использует банковские приложения и уведомления",
        interests=["автомобиль", "технологии", "инвестиции", "фитнес"], geography="Казань",
        cifra_relationship="зарплатная карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="доход стабильный, но расходы на автомобиль возникают неравномерно",
        credit_attitude="использует кредит только при понятном сроке и полном контроле долга",
    ),
    SyntheticPersona(
        persona_id="persona_003",
        label="Семейная бухгалтер с консервативным бюджетом",
        gender="женщина", age=42, monthly_income_rub=130_000,
        occupation="бухгалтер", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="Xiaomi Redmi Note 13", operating_system="Android",
        digital_behavior="пользуется приложениями, но перепроверяет тарифы и документы",
        interests=["семья", "дом", "фитнес", "телесериалы"], geography="Екатеринбург",
        cifra_relationship="не является клиентом Цифра банка",
        has_cifra_debit_or_salary_card=False,
        financial_behavior="планирует семейные расходы и держит небольшой резерв",
        credit_attitude="осторожна, опасается комиссий и сложных правил льготного периода",
    ),
    SyntheticPersona(
        persona_id="persona_004",
        label="Региональный менеджер в разъездах",
        gender="мужчина", age=51, monthly_income_rub=155_000,
        occupation="региональный менеджер по продажам", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="iPhone 13", operating_system="iOS",
        digital_behavior="часто платит телефоном и решает вопросы банка дистанционно",
        interests=["автомобиль", "телевизор", "футбол", "поездки"], geography="Нижний Новгород",
        cifra_relationship="есть дебетовая карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="бывают кассовые разрывы между командировочными расходами и компенсацией",
        credit_attitude="рассматривает лимит как резерв, не хочет носить вторую карту",
    ),
    SyntheticPersona(
        persona_id="persona_005",
        label="Работающая преподавательница старшего возраста",
        gender="женщина", age=63, monthly_income_rub=75_000,
        occupation="преподаватель колледжа", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="Realme C55", operating_system="Android",
        digital_behavior="использует базовые функции банка, сложные настройки вызывают вопросы",
        interests=["телевизор", "сад", "здоровье", "пешие прогулки"], geography="Самара",
        cifra_relationship="зарплатная карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="предпочитает заранее накопить, но иногда помогает семье до зарплаты",
        credit_attitude="к кредиту относится настороженно и нуждается в простом объяснении",
    ),
    SyntheticPersona(
        persona_id="persona_006",
        label="Молодой специалист и геймер",
        gender="мужчина", age=23, monthly_income_rub=85_000,
        occupation="специалист технической поддержки", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="Google Pixel 8", operating_system="Android",
        digital_behavior="быстро осваивает приложения, предпочитает короткие интерфейсы",
        interests=["игры", "гаджеты", "фитнес", "каршеринг"], geography="Новосибирск",
        cifra_relationship="не является клиентом Цифра банка",
        has_cifra_debit_or_salary_card=False,
        financial_behavior="регулярные расходы контролирует, крупные покупки планирует не всегда",
        credit_attitude="интересуется гибкостью, но не хочет формировать привычку жить в долг",
    ),
    SyntheticPersona(
        persona_id="persona_007",
        label="Руководительница digital-команды",
        gender="женщина", age=34, monthly_income_rub=290_000,
        occupation="руководитель digital-команды", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="iPhone 16 Pro", operating_system="iOS",
        digital_behavior="ожидает полностью цифровой путь и прозрачные уведомления",
        interests=["фитнес", "путешествия", "инвестиции", "технологии"], geography="Санкт-Петербург",
        cifra_relationship="есть дебетовая карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="есть резерв, кредитные продукты использует для управления ликвидностью",
        credit_attitude="готова пользоваться при высокой прозрачности, но небольшой лимит может быть нерелевантен",
    ),
    SyntheticPersona(
        persona_id="persona_008",
        label="Инженер с семейным бюджетом",
        gender="мужчина", age=46, monthly_income_rub=145_000,
        occupation="инженер производственного предприятия", employment_type="постоянная работа",
        citizenship="Россия", permanent_registration_country="Россия",
        device="Honor 200", operating_system="Android",
        digital_behavior="регулярно использует зарплатное банковское приложение",
        interests=["автомобиль", "ремонт дома", "хоккей", "телевизор"], geography="Челябинск",
        cifra_relationship="зарплатная карта Цифра банка",
        has_cifra_debit_or_salary_card=True,
        financial_behavior="доход предсказуемый, но ремонт автомобиля или дома создает разовые пики расходов",
        credit_attitude="готов держать кредитный лимит как запас, если правила погашения понятны",
    ),
]

In [ ]:
assert len(PERSONAS) == 8
assert len({p.persona_id for p in PERSONAS}) == 8

In [ ]:
persona_table = pd.DataFrame([
    {
        "id": p.persona_id,
        "профиль": p.label,
        "пол": p.gender,
        "возраст": p.age,
        "доход, ₽/мес": p.monthly_income_rub,
        "профессия": p.occupation,
        "девайс": p.device,
        "интересы": ", ".join(p.interests),
        "отношения с банком": p.cifra_relationship,
    }
    for p in PERSONAS
])
display(persona_table)

In [ ]:
"""## 6. Вопросы для синтетического интервью

Вопросы покрывают десять разных измерений. Мы разделяем:

- **проблему** — ситуацию или задачу человека;
- **боль** — негативное переживание или последствие проблемы;
- **потребность** — что функционально необходимо;
- **желание** — какой результат человек хочет получить;
- **возражение** — почему он может отказаться;
- **триггер** — какое событие делает продукт актуальным;
- **критерий решения** — по чему сравнивает варианты;
- **сценарий** — где именно продукт может использоваться;
- **канал** — как удобнее получить предложение;
- **действие** — насколько человек готов перейти к следующему шагу.

"""

In [ ]:
INTERVIEW_QUESTIONS = [
    InterviewQuestion(
        question_id="Q01", dimension="problem",
        text="Как вы обычно поступаете, если важный расход возникает до поступления зарплаты или компенсации?",
    ),
    InterviewQuestion(
        question_id="Q02", dimension="pain",
        text="Что в такой ситуации раздражает, тревожит или создает наибольшие неудобства?",
    ),
    InterviewQuestion(
        question_id="Q03", dimension="need",
        text="Какой финансовый инструмент или функция помогли бы решить эту ситуацию безопаснее и удобнее?",
    ),
    InterviewQuestion(
        question_id="Q04", dimension="desire",
        text="Какой результат вы хотели бы получить от резервного кредитного лимита?",
    ),
    InterviewQuestion(
        question_id="Q05", dimension="objection",
        text="Что может заставить вас отказаться от подключения такого лимита?",
    ),
    InterviewQuestion(
        question_id="Q06", dimension="trigger",
        text="После какого события или в какой момент предложение стало бы для вас особенно актуальным?",
    ),
    InterviewQuestion(
        question_id="Q07", dimension="decision_factor",
        text="Какие условия вы проверите перед подключением и какие из них будут решающими?",
    ),
    InterviewQuestion(
        question_id="Q08", dimension="use_case",
        text="Для каких конкретных категорий расходов вы могли бы использовать этот продукт?",
    ),
    InterviewQuestion(
        question_id="Q09", dimension="channel",
        text="Где и в каком формате вам удобнее узнать о продукте и разобраться в условиях?",
    ),
    InterviewQuestion(
        question_id="Q10", dimension="action",
        text="Какое действие вы готовы совершить после первого сообщения: закрыть, узнать условия, рассчитать лимит или подать заявление — и почему?",
    ),
]

In [ ]:
assert len(INTERVIEW_QUESTIONS) == 10
assert len({q.question_id for q in INTERVIEW_QUESTIONS}) == 10

In [ ]:
display(pd.DataFrame([q.model_dump() for q in INTERVIEW_QUESTIONS]))

In [ ]:
"""## 7. Persona Interviewer

Модель отвечает строго от лица заданного профиля. Она не должна добавлять новые биографические факты или выдавать симуляцию за реальное исследование.

"""

In [ ]:
def build_interview_prompt(
    product: Product,
    persona: SyntheticPersona,
    questions: list[InterviewQuestion],
) -> str:
    return f"""
Ты — модуль Synthetic Persona Interviewer системы Hypothesis Lab.

Смоделируй ответы РОВНО на 10 вопросов от лица заданной синтетической персоны.
После ответов структурируй выявленные проблемы, боли, потребности, желания,
возражения, триггеры, критерии решения и сценарии использования.

ПРАВИЛА:
1. Используй только поля персоны и подтвержденные facts продукта.
2. Не придумывай имя, работодателя, семью, кредитную историю и события прошлого.
3. Не утверждай, что доход или пол гарантируют одобрение кредита.
4. Если персона не является клиентом банка, учитывай дополнительный шаг открытия карты.
5. Различай problem, pain, need и desire по смыслу.
6. Ответь на каждый question_id ровно один раз и сохрани порядок Q01–Q10.
7. Не добавляй свойства, ставки, сроки и гарантии, которых нет в facts.
8. Не делай вывод на основании гендерных стереотипов.
9. Пиши по-русски. Верни только JSON.
10. Верни ровно 10 interview_answers; для problems, pains, needs, desires и decision_factors
    используй от 2 до 5 пунктов, а для objections, decision_triggers и relevant_use_cases — от 1 до 4.

ПРОДУКТ:
{product.model_dump_json(indent=2)}

ПЕРСОНА:
{persona.model_dump_json(indent=2)}

ВОПРОСЫ:
{json.dumps([q.model_dump() for q in questions], ensure_ascii=False, indent=2)}

ФОРМАТ ОТВЕТА (замени многоточия содержательным текстом):
{{
  "persona_id": "{persona.persona_id}",
  "interview_answers": [
    {{"question_id": "Q01", "answer": "..."}},
    {{"question_id": "Q02", "answer": "..."}},
    {{"question_id": "Q03", "answer": "..."}},
    {{"question_id": "Q04", "answer": "..."}},
    {{"question_id": "Q05", "answer": "..."}},
    {{"question_id": "Q06", "answer": "..."}},
    {{"question_id": "Q07", "answer": "..."}},
    {{"question_id": "Q08", "answer": "..."}},
    {{"question_id": "Q09", "answer": "..."}},
    {{"question_id": "Q10", "answer": "..."}}
  ],
  "problems": ["...", "..."],
  "pains": ["...", "..."],
  "needs": ["...", "..."],
  "desires": ["...", "..."],
  "objections": ["..."],
  "decision_triggers": ["..."],
  "decision_factors": ["...", "..."],
  "relevant_use_cases": ["..."],
  "best_message_angle": "...",
  "fit_summary": "..."
}}
""".strip()

In [ ]:
def validate_research_questions(research: PersonaResearch):
    expected = [q.question_id for q in INTERVIEW_QUESTIONS]
    actual = [a.question_id for a in research.interview_answers]
    if actual != expected:
        raise ValueError(
            f"{research.persona_id}: неверный набор/порядок вопросов {actual}; ожидался {expected}"
        )

In [ ]:
def interview_persona(product: Product, persona: SyntheticPersona) -> PersonaResearch:
    result = run_llm_json(
        system_prompt=(
            "Ты проводишь контролируемую симуляцию интервью синтетической персоны. "
            "Не выдавай предположения за реальные данные."
        ),
        user_prompt=build_interview_prompt(product, persona, INTERVIEW_QUESTIONS),
        response_model=PersonaResearch,
        max_attempts=5,
        temperature=0.15,
        max_new_tokens=2300,
    )
    if result.persona_id != persona.persona_id:
        raise ValueError(
            f"Interviewer вернул {result.persona_id}, ожидался {persona.persona_id}"
        )
    validate_research_questions(result)
    return result

In [ ]:
persona_research = []
for index, persona in enumerate(PERSONAS, start=1):
    print(f"Интервью {index}/8: {persona.persona_id} — {persona.label}")
    persona_research.append(interview_persona(product, persona))

In [ ]:
print("Получено интервью:", len(persona_research))

In [ ]:
"""## 8. Eligibility Gate

Python проверяет формальные критерии продукта до ранжирования. Это не банковский скоринг и не прогноз одобрения. Проверка лишь исключает профили, которые явно не соответствуют заданным пользователем правилам.

Наличие карты Цифра банка учитывается как трение в `adoption_readiness`, но не исключает человека: он может сначала открыть дебетовую карту.

"""

In [ ]:
def check_rule(persona: SyntheticPersona, rule: EligibilityRule) -> bool:
    actual = getattr(persona, rule.field)
    expected = rule.value

    if rule.operator == "eq":
        return actual == expected
    if rule.operator == "in":
        return actual in expected
    if rule.operator == "gte":
        return actual >= expected
    if rule.operator == "lte":
        return actual <= expected
    if rule.operator == "between":
        low, high = expected
        return low <= actual <= high
    if rule.operator == "truthy":
        return bool(actual)
    raise ValueError(f"Неизвестный operator: {rule.operator}")

In [ ]:
def eligibility_report(product: Product, persona: SyntheticPersona) -> dict:
    checks = [
        {
            "rule_id": rule.rule_id,
            "description": rule.description,
            "passed": check_rule(persona, rule),
        }
        for rule in product.eligibility_rules
    ]
    return {
        "persona_id": persona.persona_id,
        "hard_eligible": all(item["passed"] for item in checks),
        "checks": checks,
        "has_required_card_now": persona.has_cifra_debit_or_salary_card,
    }

In [ ]:
eligibility = {
    p.persona_id: eligibility_report(product, p)
    for p in PERSONAS
}

In [ ]:
display(pd.DataFrame([
    {
        "persona_id": p.persona_id,
        "hard_eligible": eligibility[p.persona_id]["hard_eligible"],
        "есть карта Цифра банка": eligibility[p.persona_id]["has_required_card_now"],
        "пройдено правил": sum(x["passed"] for x in eligibility[p.persona_id]["checks"]),
        "всего правил": len(eligibility[p.persona_id]["checks"]),
    }
    for p in PERSONAS
]))

In [ ]:
"""## 9. Persona–Product Critic

Каждая персона оценивается отдельно, без сравнения с остальными. Метрики:

| Метрика | Вес | Что измеряет |
|---|---:|---|
| `need_strength` | 25% | Насколько выражена проблема или триггер |
| `product_value_fit` | 30% | Насколько подтверждённые свойства продукта решают задачу |
| `adoption_readiness` | 20% | Готовность подключить продукт с учётом возражений и текущих отношений с банком |
| `digital_access_fit` | 10% | Возможность и удобство пройти мобильный путь |
| `hypothesis_testability` | 15% | Есть ли конкретный сценарий, сообщение и измеримое действие |

Гендер не оценивается. Доход может объяснять контекст расходов, но не используется как имитация кредитного решения банка.

"""

In [ ]:
PERSONA_SCORE_WEIGHTS = {
    "need_strength": 0.25,
    "product_value_fit": 0.30,
    "adoption_readiness": 0.20,
    "digital_access_fit": 0.10,
    "hypothesis_testability": 0.15,
}

In [ ]:
def build_persona_score_prompt(
    product: Product,
    persona: SyntheticPersona,
    research: PersonaResearch,
    eligibility_item: dict,
) -> str:
    return f"""
Ты — строгий Persona–Product Critic системы Hypothesis Lab.

Оцени ОДНУ персону для ОДНОГО продукта. Не сравнивай ее с другими персонами.

ШКАЛА:
0–39 = слабое соответствие
40–59 = ниже среднего
60–74 = рабочее соответствие
75–84 = хорошее соответствие
85–94 = очень сильное соответствие
95–100 = исключительное; использовать редко

МЕТРИКИ:
- need_strength: выраженность проблемы/триггера;
- product_value_fit: соответствие только подтвержденным facts;
- adoption_readiness: готовность подключить с учетом возражений и наличия карты банка;
- digital_access_fit: соответствие устройства и привычек мобильному пути;
- hypothesis_testability: конкретность сценария, сообщения и измеримого действия.

ПРАВИЛА:
1. Оценивай только эту персону. Итоговый score и место НЕ считай.
2. Не используй пол как основание оценки.
3. Не делай вывод об одобрении кредита по доходу, профессии или возрасту.
4. Учитывай только подтвержденные свойства продукта.
5. Наличие карты банка влияет на трение подключения, а не на силу потребности.
6. Evidence должно ссылаться на конкретные поля персоны или результаты интервью.
7. Не завышай оценки: обычный хороший профиль — примерно 65–82.
8. Пиши по-русски. Верни только JSON.
9. В evidence используй от 2 до 5 конкретных пунктов.

ПРОДУКТ:
{product.model_dump_json(indent=2)}

ПЕРСОНА:
{persona.model_dump_json(indent=2)}

РЕЗУЛЬТАТ СИНТЕТИЧЕСКОГО ИНТЕРВЬЮ:
{research.model_dump_json(indent=2)}

ФОРМАЛЬНАЯ ПРОВЕРКА PYTHON:
{json.dumps(eligibility_item, ensure_ascii=False, indent=2)}

ФОРМАТ ОТВЕТА:
{{
  "persona_id": "{persona.persona_id}",
  "need_strength": 70,
  "product_value_fit": 70,
  "adoption_readiness": 70,
  "digital_access_fit": 70,
  "hypothesis_testability": 70,
  "strongest_signal": "...",
  "main_barrier": "...",
  "evidence": ["...", "..."],
  "explanation": "..."
}}
""".strip()

In [ ]:
def score_persona(
    product: Product,
    persona: SyntheticPersona,
    research: PersonaResearch,
) -> PersonaMetricScores:
    result = run_llm_json(
        system_prompt=(
            "Ты независимый строгий оценщик соответствия персоны продукту. "
            "Не ранжируй список и не имитируй банковский кредитный скоринг."
        ),
        user_prompt=build_persona_score_prompt(
            product, persona, research, eligibility[persona.persona_id]
        ),
        response_model=PersonaMetricScores,
        max_attempts=5,
        temperature=0.10,
        max_new_tokens=1100,
    )
    if result.persona_id != persona.persona_id:
        raise ValueError(
            f"Critic вернул {result.persona_id}, ожидался {persona.persona_id}"
        )
    return result

In [ ]:
persona_scores = []
research_by_id = {item.persona_id: item for item in persona_research}

In [ ]:
for index, persona in enumerate(PERSONAS, start=1):
    print(f"Оценка {index}/8: {persona.persona_id}")
    persona_scores.append(
        score_persona(product, persona, research_by_id[persona.persona_id])
    )

In [ ]:
"""## 10. Weighted Ranker на Python и выбор топ‑3

LLM не видит оценки других персон и не выбирает победителей. Код применяет фиксированные веса, исключает только формально неподходящие профили и возвращает ровно три персоны.

"""

In [ ]:
def calculate_persona_score(scores: PersonaMetricScores) -> float:
    value = sum(
        getattr(scores, metric) * weight
        for metric, weight in PERSONA_SCORE_WEIGHTS.items()
    )
    return round(value, 2)

In [ ]:
def rank_personas(
    personas: list[SyntheticPersona],
    scores: list[PersonaMetricScores],
    eligibility_map: dict[str, dict],
) -> list[dict]:
    persona_by_id = {p.persona_id: p for p in personas}
    rows = []

    for item in scores:
        eligibility_item = eligibility_map[item.persona_id]
        rows.append({
            "persona_id": item.persona_id,
            "label": persona_by_id[item.persona_id].label,
            "hard_eligible": eligibility_item["hard_eligible"],
            "has_required_card_now": eligibility_item["has_required_card_now"],
            "metrics": item.model_dump(),
            "weights": PERSONA_SCORE_WEIGHTS,
            "final_score": calculate_persona_score(item),
        })

    rows.sort(
        key=lambda x: (
            not x["hard_eligible"],
            -x["final_score"],
            x["persona_id"],
        )
    )

    eligible_rank = 0
    for row in rows:
        if row["hard_eligible"]:
            eligible_rank += 1
            row["rank"] = eligible_rank
        else:
            row["rank"] = None
    return rows

In [ ]:
persona_ranking = rank_personas(PERSONAS, persona_scores, eligibility)
top_3 = [row for row in persona_ranking if row["hard_eligible"]][:3]

In [ ]:
if len(top_3) != 3:
    raise RuntimeError(
        f"После формальной проверки доступно только {len(top_3)} персон; нужно минимум 3."
    )

In [ ]:
ranking_table = pd.DataFrame([
    {
        "место": row["rank"],
        "persona_id": row["persona_id"],
        "персона": row["label"],
        "score": row["final_score"],
        "потребность": row["metrics"]["need_strength"],
        "fit продукта": row["metrics"]["product_value_fit"],
        "готовность": row["metrics"]["adoption_readiness"],
        "digital fit": row["metrics"]["digital_access_fit"],
        "тестируемость": row["metrics"]["hypothesis_testability"],
        "главный сигнал": row["metrics"]["strongest_signal"],
        "главный барьер": row["metrics"]["main_barrier"],
    }
    for row in persona_ranking
])
display(ranking_table)

In [ ]:
print("ТОП-3:")
for row in top_3:
    print(f'{row["rank"]}. {row["persona_id"]} — {row["label"]}: {row["final_score"]}/100')

In [ ]:
"""## 11. Генератор гипотез для топ‑3

Для каждой выбранной персоны создаются три **разные** гипотезы. Каждая содержит:

- инсайт из интервью;
- формулировку «если → то → потому что»;
- сообщение, канал, CTA и дизайн теста;
- основную метрику;
- `why_for_persona` — почему гипотеза подходит именно этой персоне;
- ссылки на подтверждённые `fact_id`;
- риски и ограничения.

"""

In [ ]:
def build_hypothesis_prompt(
    product: Product,
    persona: SyntheticPersona,
    research: PersonaResearch,
    ranking_item: dict,
) -> str:
    expected_ids = [
        f"{persona.persona_id}_hyp_01",
        f"{persona.persona_id}_hyp_02",
        f"{persona.persona_id}_hyp_03",
    ]
    template_variants = [
        ("pain", "safety_net"),
        ("desire", "control"),
        ("objection", "clarity"),
    ]
    hypothesis_template = {
        "hypotheses": [
            {
                "hypothesis_id": hypothesis_id,
                "persona_id": persona.persona_id,
                "insight_type": insight_type,
                "insight_basis": ["..."],
                "value_angle": value_angle,
                "hypothesis_statement": "Если ..., то ... вырастет, потому что ...",
                "headline": "...",
                "message_frame": "...",
                "cta": "...",
                "channel": "push",
                "selected_fact_ids": ["F1"],
                "experiment_design": "...",
                "primary_metric": "application_start_rate",
                "why_for_persona": "...",
                "risks": ["..."],
            }
            for hypothesis_id, (insight_type, value_angle)
            in zip(expected_ids, template_variants)
        ]
    }
    return f"""
Ты — Marketing Hypothesis Generator системы Hypothesis Lab.

Создай РОВНО 3 разные гипотезы для выбранной персоны и продукта.

ПРАВИЛА:
1. Каждая гипотеза должна опираться минимум на один конкретный инсайт интервью.
2. hypothesis_statement использует логику: «Если [воздействие], то [измеримое действие/метрика] вырастет, потому что [инсайт]».
3. Не указывай выдуманный процент роста или гарантированный результат. Не используй
   числа, кроме чисел, буквально присутствующих в facts продукта.
4. В трех гипотезах должны различаться value_angle и основной исследовательский инсайт.
5. Выбирай только разрешенные selected_fact_ids из продукта.
6. Не добавляй ставок, сроков, лимитов, комиссий, функций и гарантий вне facts.
7. Не обещай одобрение кредита и не используй запрещенные claims.
8. why_for_persona должен прямо связать гипотезу с болью, желанием, возражением,
   триггером или сценарием именно этой персоны.
9. Канал должен входить в allowed_channels продукта и учитывать устройство/привычки.
10. primary_metric — одна измеримая метрика, например application_start_rate,
    application_submit_rate, contract_sign_rate, CTR или details_open_rate.
11. Пиши по-русски. Верни только JSON.
12. Идентификаторы по порядку: {expected_ids}.

ПРОДУКТ:
{product.model_dump_json(indent=2)}

ПЕРСОНА:
{persona.model_dump_json(indent=2)}

ИНТЕРВЬЮ:
{research.model_dump_json(indent=2)}

ОЦЕНКА И МЕСТО:
{json.dumps(ranking_item, ensure_ascii=False, indent=2)}

ФОРМАТ ОТВЕТА:
{json.dumps(hypothesis_template, ensure_ascii=False, indent=2)}
""".strip()

In [ ]:
def generated_text_blob(hypothesis: HypothesisDraft) -> str:
    return " ".join([
        hypothesis.hypothesis_statement,
        hypothesis.headline,
        hypothesis.message_frame,
        hypothesis.cta,
        hypothesis.experiment_design,
        hypothesis.why_for_persona,
    ])

In [ ]:
def extract_numeric_claims(text: str) -> set[str]:
    return set(re.findall(r"(?<![A-Za-zА-Яа-я_])\d[\d\s]*(?:[.,]\d+)?\s*%?", text))

In [ ]:
def compact_number(token: str) -> str:
    return re.sub(r"\s+", "", token).replace(",", ".")

In [ ]:
def validate_hypothesis_batch(
    product: Product,
    persona: SyntheticPersona,
    batch: HypothesisBatch,
):
    errors = []
    expected_ids = [
        f"{persona.persona_id}_hyp_01",
        f"{persona.persona_id}_hyp_02",
        f"{persona.persona_id}_hyp_03",
    ]
    actual_ids = [h.hypothesis_id for h in batch.hypotheses]
    if actual_ids != expected_ids:
        errors.append(f"Неверные hypothesis_id: {actual_ids}")

    if any(h.persona_id != persona.persona_id for h in batch.hypotheses):
        errors.append("В batch есть гипотеза другой персоны.")

    allowed_fact_ids = {f.fact_id for f in product.facts}
    allowed_channels = set(product.allowed_channels)
    allowed_numbers = {
        compact_number(token)
        for fact in product.facts
        for token in extract_numeric_claims(fact.text)
    }

    value_angles = [h.value_angle for h in batch.hypotheses]
    if len(set(value_angles)) != 3:
        errors.append("У трех гипотез должны быть разные value_angle.")

    normalized_headlines = [re.sub(r"\W+", " ", h.headline.lower()).strip() for h in batch.hypotheses]
    if len(set(normalized_headlines)) != 3:
        errors.append("Заголовки гипотез дублируются.")

    for h in batch.hypotheses:
        unknown_facts = set(h.selected_fact_ids) - allowed_fact_ids
        if unknown_facts:
            errors.append(f"{h.hypothesis_id}: неизвестные fact_id {sorted(unknown_facts)}")
        if h.channel not in allowed_channels:
            errors.append(f"{h.hypothesis_id}: запрещенный канал {h.channel}")

        text = generated_text_blob(h).lower()
        for phrase in product.forbidden_claims:
            if phrase.lower() in text:
                errors.append(f"{h.hypothesis_id}: запрещенная формулировка '{phrase}'")

        used_numbers = {compact_number(x) for x in extract_numeric_claims(text)}
        unknown_numbers = used_numbers - allowed_numbers
        if unknown_numbers:
            errors.append(
                f"{h.hypothesis_id}: числа без подтверждения в facts {sorted(unknown_numbers)}"
            )

        if "потому что" not in h.hypothesis_statement.lower():
            errors.append(f"{h.hypothesis_id}: нет части «потому что».")

    if errors:
        raise ValueError("Hypothesis Validator:\n- " + "\n- ".join(errors))

In [ ]:
def generate_hypotheses_for_persona(
    product: Product,
    persona: SyntheticPersona,
    research: PersonaResearch,
    ranking_item: dict,
    max_batches: int = 3,
) -> HypothesisBatch:
    last_error = None
    for attempt in range(1, max_batches + 1):
        print(f"  batch {attempt}/{max_batches}")
        batch = run_llm_json(
            system_prompt=(
                "Ты создаешь проверяемые маркетинговые гипотезы на основе "
                "синтетического интервью и подтвержденных продуктовых фактов."
            ),
            user_prompt=build_hypothesis_prompt(product, persona, research, ranking_item),
            response_model=HypothesisBatch,
            temperature=0.35,
            max_new_tokens=2000,
            retry_feedback=str(last_error) if last_error else None,
        )
        try:
            validate_hypothesis_batch(product, persona, batch)
            return batch
        except Exception as exc:
            last_error = exc
            print("  batch отклонен:", exc)
    raise RuntimeError(
        f"Не удалось получить валидные гипотезы для {persona.persona_id}: {last_error}"
    )

In [ ]:
def materialize_hypothesis(product: Product, draft: HypothesisDraft) -> dict:
    fact_map = {f.fact_id: f.text for f in product.facts}
    result = draft.model_dump()
    result["proof_points"] = [fact_map[fact_id] for fact_id in draft.selected_fact_ids]
    result["status"] = "synthetic_hypothesis_requires_real_validation"
    return result

In [ ]:
def hypothesis_tokens(item: dict) -> set[str]:
    text = " ".join([
        item["hypothesis_statement"],
        item["headline"],
        item["message_frame"],
        item["why_for_persona"],
    ]).lower()
    return {
        token
        for token in re.findall(r"[a-zа-яё]{4,}", text)
        if token not in {"если", "потому", "который", "которая", "через", "этого"}
    }

In [ ]:
def cross_persona_diversity_warnings(hypotheses_map: dict[str, list[dict]]) -> list[str]:
    items = [item for batch in hypotheses_map.values() for item in batch]
    warnings = []
    for index, left in enumerate(items):
        for right in items[index + 1:]:
            if left["persona_id"] == right["persona_id"]:
                continue
            left_tokens = hypothesis_tokens(left)
            right_tokens = hypothesis_tokens(right)
            union = left_tokens | right_tokens
            similarity = len(left_tokens & right_tokens) / len(union) if union else 0.0
            if similarity >= 0.80:
                warnings.append(
                    f'{left["hypothesis_id"]} и {right["hypothesis_id"]}: '
                    f'возможный смысловой дубль, Jaccard={similarity:.2f}'
                )
    return warnings

In [ ]:
persona_by_id = {p.persona_id: p for p in PERSONAS}
hypotheses_by_persona = {}

In [ ]:
for index, row in enumerate(top_3, start=1):
    persona = persona_by_id[row["persona_id"]]
    print(f"Гипотезы {index}/3: {persona.persona_id} — {persona.label}")
    batch = generate_hypotheses_for_persona(
        product,
        persona,
        research_by_id[persona.persona_id],
        row,
    )
    hypotheses_by_persona[persona.persona_id] = [
        materialize_hypothesis(product, h)
        for h in batch.hypotheses
    ]

In [ ]:
assert sum(len(items) for items in hypotheses_by_persona.values()) == 9

In [ ]:
diversity_warnings = cross_persona_diversity_warnings(hypotheses_by_persona)
if diversity_warnings:
    print("Предупреждения Cross-Persona Diversity Checker:")
    for warning in diversity_warnings:
        print("-", warning)
else:
    print("Cross-Persona Diversity Checker: явных дублей не найдено.")

In [ ]:
"""## 12. Итоговый вывод: топ‑3 и по 3 гипотезы

"""

In [ ]:
for row in top_3:
    persona_id = row["persona_id"]
    research = research_by_id[persona_id]
    print("=" * 100)
    print(f'МЕСТО {row["rank"]}: {row["label"]} — {row["final_score"]}/100')
    print("Почему в топе:", row["metrics"]["explanation"])
    print("Главный сигнал:", row["metrics"]["strongest_signal"])
    print("Главный барьер:", row["metrics"]["main_barrier"])
    print("Ключевая боль:", research.pains[0])
    print("Ключевое желание:", research.desires[0])

    for hyp in hypotheses_by_persona[persona_id]:
        print("\n", hyp["hypothesis_id"], "—", hyp["headline"])
        print("Гипотеза:", hyp["hypothesis_statement"])
        print("Почему подходит:", hyp["why_for_persona"])
        print("Канал / метрика:", hyp["channel"], "/", hyp["primary_metric"])
        print("Подтвержденные факты:")
        for fact in hyp["proof_points"]:
            print(" -", fact)

In [ ]:
hypothesis_table = pd.DataFrame([
    {
        "место персоны": row["rank"],
        "persona_id": row["persona_id"],
        "персона": row["label"],
        "hypothesis_id": hyp["hypothesis_id"],
        "угол": hyp["value_angle"],
        "заголовок": hyp["headline"],
        "канал": hyp["channel"],
        "метрика": hyp["primary_metric"],
        "почему подходит": hyp["why_for_persona"],
    }
    for row in top_3
    for hyp in hypotheses_by_persona[row["persona_id"]]
])
display(hypothesis_table)

In [ ]:
"""## 13. Сохранение результатов

Сохраняются три специализированных JSON и один полный файл:

- интервью всех 8 персон;
- рейтинг и топ‑3;
- 9 гипотез;
- полный результат одного запуска.

"""

In [ ]:
run_id = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_utc")
OUTPUT_DIR = Path("/content/hypothesis_lab_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
common_meta = {
    "run_id": run_id,
    "provider": "local_colab",
    "model": MODEL_NAME,
    "pipeline_version": "persona_ranking_hypotheses_v4",
    "synthetic_data_warning": (
        "Результаты основаны на синтетических персонах и требуют проверки "
        "на реальных клиентах до принятия продуктовых решений."
    ),
}

In [ ]:
research_output = {
    **common_meta,
    "product": product.model_dump(),
    "readiness_questions": readiness_questions,
    "interview_questions": [q.model_dump() for q in INTERVIEW_QUESTIONS],
    "personas": [p.model_dump() for p in PERSONAS],
    "persona_research": [r.model_dump() for r in persona_research],
}

In [ ]:
ranking_output = {
    **common_meta,
    "product": product.model_dump(),
    "metric_weights": PERSONA_SCORE_WEIGHTS,
    "eligibility": eligibility,
    "ranking": persona_ranking,
    "top_3": top_3,
}

In [ ]:
hypotheses_output = {
    **common_meta,
    "product": product.model_dump(),
    "top_3": top_3,
    "hypotheses_by_persona": hypotheses_by_persona,
    "hypotheses_count": sum(len(v) for v in hypotheses_by_persona.values()),
    "diversity_warnings": diversity_warnings,
}

In [ ]:
full_output = {
    **common_meta,
    "product": product.model_dump(),
    "interview_questions": [q.model_dump() for q in INTERVIEW_QUESTIONS],
    "personas": [p.model_dump() for p in PERSONAS],
    "persona_research": [r.model_dump() for r in persona_research],
    "metric_weights": PERSONA_SCORE_WEIGHTS,
    "eligibility": eligibility,
    "ranking": persona_ranking,
    "top_3": top_3,
    "hypotheses_by_persona": hypotheses_by_persona,
    "diversity_warnings": diversity_warnings,
}

In [ ]:
OUTPUT_FILES = {
    "research": OUTPUT_DIR / "hypothesis_lab_persona_research_v4.json",
    "ranking": OUTPUT_DIR / "hypothesis_lab_persona_ranking_v4.json",
    "hypotheses": OUTPUT_DIR / "hypothesis_lab_persona_hypotheses_v4.json",
    "full": OUTPUT_DIR / "hypothesis_lab_full_result_v4.json",
}

In [ ]:
for key, payload in {
    "research": research_output,
    "ranking": ranking_output,
    "hypotheses": hypotheses_output,
    "full": full_output,
}.items():
    with OUTPUT_FILES[key].open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

In [ ]:
print("Сохранено:")
for path in OUTPUT_FILES.values():
    print(path)

In [ ]:
from google.colab import files

In [ ]:
for path in OUTPUT_FILES.values():
    files.download(str(path))

In [ ]:
"""## 14. Критерии успешного прогона

После полного запуска должно выполняться:

- обработано ровно 8 фиксированных персон;
- каждая ответила на Q01–Q10;
- для каждой есть пять независимых оценок 0–100;
- Python вернул ровно 3 формально подходящие персоны;
- для каждой топ‑персоны создано ровно 3 гипотезы, всего 9;
- внутри одной персоны не повторяются `value_angle` и заголовки;
- между персонами выполняется дополнительная проверка на близкие смысловые дубли;
- все гипотезы ссылаются только на существующие `fact_id`;
- числа в сообщениях подтверждены продуктовыми фактами;
- нет обещаний гарантированного одобрения или других запрещённых формулировок;
- каждая гипотеза содержит явное объяснение `why_for_persona`;
- в JSON есть предупреждение, что синтетические выводы требуют реальной проверки.

"""